In [1]:
# ── Standard ──────────────────────────────────────────────────────────────────
import os
import re
import json
import time
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", category=UserWarning)

# ── Audio ─────────────────────────────────────────────────────────────────────
import librosa
import soundfile as sf
import numpy as np

# ── HuggingFace ───────────────────────────────────────────────────────────────
import torch
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    pipeline,
)

# ── Metrics ───────────────────────────────────────────────────────────────────
from jiwer import wer, cer
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# ── Data ──────────────────────────────────────────────────────────────────────
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use("dark_background")
pd.set_option("display.max_colwidth", 120)

# ── Paths ─────────────────────────────────────────────────────────────────────
PROJECT_ROOT = Path("C:/xcas-ga-comms-assistant")
AUDIO_DIR    = PROJECT_ROOT / "tartan_data/kbtp/2020/10/10-22-20_audio"
CALLOUTS_CSV = PROJECT_ROOT / "data/interim/adsb_with_callouts_2020-10-22.csv"
MANIFEST_CSV = PROJECT_ROOT / "data/interim/manifest_2020-10-22.csv"
OUTPUT_DIR   = PROJECT_ROOT / "data/interim"
RESULTS_DIR  = PROJECT_ROOT / "outputs"
RESULTS_DIR.mkdir(exist_ok=True)

# ── GPU Setup ─────────────────────────────────────────────────────────────────
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("═"*55)
print("PROJECT 2: ASR BASELINE — Whisper ATC Fine-tuned")
print("═"*55)
print(f"\nDevice          : {DEVICE.upper()}")

if DEVICE == "cuda":
    gpu = torch.cuda.get_device_properties(0)
    vram_gb = gpu.total_memory / 1e9
    print(f"GPU             : {gpu.name}")
    print(f"VRAM            : {vram_gb:.1f} GB")
    print(f"CUDA version    : {torch.version.cuda}")
    
    # Whisper model size guide for your 12GB VRAM:
    # tiny   (39M params)  → ~0.5 GB  → ~32× real-time
    # small  (244M params) → ~1.0 GB  → ~8× real-time   ← our start
    # medium (769M params) → ~3.0 GB  → ~4× real-time
    # large  (1.5B params) → ~6.0 GB  → ~2× real-time
    # large-v3             → ~6.0 GB  → fits easily
    print(f"\nModel size guide for {vram_gb:.0f}GB VRAM:")
    models = [("tiny","39M","~0.5GB","fits"), ("small","244M","~1.0GB","fits ← start here"),
              ("medium","769M","~3.0GB","fits"), ("large-v3","1.5B","~6.0GB","fits")]
    for name, params, vram, note in models:
        print(f"  whisper-{name:8s} {params:6s} params  {vram:7s}  {note}")
else:
    print("⚠  No GPU found — running on CPU (slow but works)")

print(f"\nAudio dir       : {AUDIO_DIR}")
wav_files = sorted(AUDIO_DIR.glob("*.wav"))
txt_files = sorted(AUDIO_DIR.glob("*.txt"))
print(f"WAV files found : {len(wav_files)}")
print(f"TXT files found : {len(txt_files)}")

c:\xcas-ga-comms-assistant\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


═══════════════════════════════════════════════════════
PROJECT 2: ASR BASELINE — Whisper ATC Fine-tuned
═══════════════════════════════════════════════════════

Device          : CUDA
GPU             : NVIDIA GeForce RTX 5070 Ti Laptop GPU
VRAM            : 12.8 GB
CUDA version    : 12.8

Model size guide for 13GB VRAM:
  whisper-tiny     39M    params  ~0.5GB   fits
  whisper-small    244M   params  ~1.0GB   fits ← start here
  whisper-medium   769M   params  ~3.0GB   fits
  whisper-large-v3 1.5B   params  ~6.0GB   fits

Audio dir       : C:\xcas-ga-comms-assistant\tartan_data\kbtp\2020\10\10-22-20_audio
WAV files found : 52
TXT files found : 52


In [2]:
# ═══════════════════════════════════════════════════════════════════════════════
# AUDIO PROFILING
#
# 📚 Concept — Why audio analysis before ASR?
# Whisper was trained on audio at 16kHz, mono, 30-second chunks.
# Your TartanAviation recordings are at 44.1kHz stereo (the paper says so).
# If you feed 44.1kHz audio to a 16kHz model without resampling,
# the model "hears" everything at 2.76× normal speed — like chipmunks.
# Always profile your audio before modeling.
# ═══════════════════════════════════════════════════════════════════════════════

print("AUDIO PROFILE — First 3 WAV files")
print("="*55)

TARGET_SR = 16000   # Whisper's required sample rate

for wav_path in sorted(wav_files)[:3]:
    # librosa.load automatically resamples to target sr if specified
    # duration=30 loads only first 30 seconds for fast profiling
    audio_info, sr_native = librosa.load(str(wav_path), sr=None, duration=30)
    
    # Load full file just for duration
    full_duration = librosa.get_duration(path=str(wav_path))
    
    print(f"\nFile     : {wav_path.name}")
    print(f"Native SR: {sr_native:,} Hz  "
          f"({'✅ matches Whisper' if sr_native==TARGET_SR else f'⚠ needs resample to {TARGET_SR}Hz'})")
    print(f"Duration : {full_duration:.1f}s  ({full_duration/60:.1f} min)")
    print(f"Channels : {1 if audio_info.ndim==1 else audio_info.shape[0]}")
    print(f"Shape    : {audio_info.shape}")
    print(f"Amplitude: min={audio_info.min():.3f}  max={audio_info.max():.3f}  "
          f"rms={np.sqrt(np.mean(audio_info**2)):.4f}")

# What resampling looks like conceptually
print(f"\n📚 Resampling explained:")
print(f"   Native 44,100 Hz = 44,100 samples per second of audio")
print(f"   Whisper needs  16,000 Hz = 16,000 samples per second")
print(f"   librosa.resample() uses polyphase filtering to")
print(f"   downsample without aliasing (audio distortion)")
print(f"   After resample: same duration, fewer samples, correct pitch")

AUDIO PROFILE — First 3 WAV files

File     : 10.wav
Native SR: 44,100 Hz  (⚠ needs resample to 16000Hz)
Duration : 170.5s  (2.8 min)
Channels : 1
Shape    : (1323000,)
Amplitude: min=-0.632  max=0.604  rms=0.1327

File     : 12.wav
Native SR: 44,100 Hz  (⚠ needs resample to 16000Hz)
Duration : 900.4s  (15.0 min)
Channels : 1
Shape    : (1323000,)
Amplitude: min=-0.003  max=0.001  rms=0.0002

File     : 13.wav
Native SR: 44,100 Hz  (⚠ needs resample to 16000Hz)
Duration : 899.9s  (15.0 min)
Channels : 1
Shape    : (1323000,)
Amplitude: min=-0.621  max=0.604  rms=0.1098

📚 Resampling explained:
   Native 44,100 Hz = 44,100 samples per second of audio
   Whisper needs  16,000 Hz = 16,000 samples per second
   librosa.resample() uses polyphase filtering to
   downsample without aliasing (audio distortion)
   After resample: same duration, fewer samples, correct pitch


In [3]:
# ═══════════════════════════════════════════════════════════════════════════════
# LOAD WHISPER (ATC FINE-TUNED)
#
# 📚 Concept — Whisper Architecture:
#
#   Input audio (30s chunks)
#        ↓
#   Log-Mel Spectrogram  ← converts waveform to 80-band frequency image
#        ↓                  (80 mel bins × 3000 time frames = 30s window)
#   CNN feature extractor ← 2 conv layers, extracts local audio patterns
#        ↓
#   Transformer ENCODER  ← 6 layers (small), sees the full audio
#        ↓
#   Cross-attention      ← decoder attends to encoder representations
#        ↓
#   Transformer DECODER  ← 6 layers, autoregressively generates tokens
#        ↓
#   Text output          ← word-by-word generation, left to right
#
# 📚 Concept — Fine-tuning vs Pre-training:
#   OpenAI pre-trained Whisper on 680,000 hours of general audio.
#   jlvdoorn then FINE-TUNED it on ATCO2 (ATC speech) — meaning they
#   continued training with a small learning rate on domain-specific data.
#   The model KEEPS general speech knowledge but ADAPTS to:
#     - ATC vocabulary (callsigns, runway IDs, "cleared", "traffic")
#     - Radio noise and compression artifacts
#     - Clipped, fast, technical speech style
#
# 📚 Concept — WhisperProcessor vs WhisperForConditionalGeneration:
#   Processor = handles BOTH input (audio → features) and output (tokens → text)
#   Model     = the actual neural network weights
#   You always need both. The processor is deterministic (no weights).
# ═══════════════════════════════════════════════════════════════════════════════

MODEL_ID = "jlvdoorn/whisper-small.en-atco2-asr"

print(f"Loading model: {MODEL_ID}")
print(f"Device: {DEVICE}")
print("(First run downloads ~500MB — cached for subsequent runs)\n")

t0 = time.time()

processor = WhisperProcessor.from_pretrained(MODEL_ID)
model     = WhisperForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if DEVICE=="cuda" else torch.float32,
)
model = model.to(DEVICE)
model.eval()   # disable dropout — we're doing inference not training

load_time = time.time() - t0

print(f"✅ Model loaded in {load_time:.1f}s")

# Model stats
total_params = sum(p.numel() for p in model.parameters())
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nModel statistics:")
print(f"  Total parameters    : {total_params/1e6:.1f}M")
print(f"  Architecture        : {model.config.model_type}")
print(f"  Encoder layers      : {model.config.encoder_layers}")
print(f"  Decoder layers      : {model.config.decoder_layers}")
print(f"  Attention heads     : {model.config.encoder_attention_heads}")
print(f"  Hidden size         : {model.config.d_model}")
print(f"  Mel bins            : {model.config.num_mel_bins}")
print(f"  Max source positions: {model.config.max_source_positions} "
      f"(= {model.config.max_source_positions/100:.0f}s audio)")
print(f"  Vocab size          : {model.config.vocab_size:,} tokens")

if DEVICE == "cuda":
    vram_used = torch.cuda.memory_allocated() / 1e9
    vram_total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"\nVRAM used by model  : {vram_used:.2f} GB / {vram_total:.1f} GB")
    print(f"VRAM remaining      : {vram_total - vram_used:.2f} GB")

Loading model: jlvdoorn/whisper-small.en-atco2-asr
Device: cuda
(First run downloads ~500MB — cached for subsequent runs)



Loading weights: 100%|██████████| 479/479 [00:00<00:00, 4130.52it/s]


✅ Model loaded in 57.6s

Model statistics:
  Total parameters    : 241.7M
  Architecture        : whisper
  Encoder layers      : 12
  Decoder layers      : 12
  Attention heads     : 12
  Hidden size         : 768
  Mel bins            : 80
  Max source positions: 1500 (= 15s audio)
  Vocab size          : 51,864 tokens

VRAM used by model  : 0.50 GB / 12.8 GB
VRAM remaining      : 12.32 GB


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# TRANSCRIPTION PIPELINE
#
# 📚 Concept — Why chunk audio into 30s segments?
# Whisper's positional embeddings are trained for exactly 30-second windows.
# Audio longer than 30s must be chunked. The model doesn't know about time
# beyond 30s — it has no memory across chunks. For our recordings (up to
# 15 minutes), we chunk with overlap so words at chunk boundaries aren't lost.
#
# 📚 Concept — chunk_length_s=30, stride_length_s=5:
# stride = overlap between consecutive chunks.
# Chunk 1: seconds  0–30
# Chunk 2: seconds 25–55  (5s overlap with chunk 1)
# Chunk 3: seconds 50–80  (5s overlap with chunk 2)
# The overlapping portions are merged using a stitching algorithm.
# Without stride, words spoken at the 30s boundary get cut off.
# ═══════════════════════════════════════════════════════════════════════════════

# HuggingFace pipeline handles chunking + stitching automatically
asr_pipeline = pipeline(
    task                = "automatic-speech-recognition",
    model               = model,
    tokenizer           = processor.tokenizer,
    feature_extractor   = processor.feature_extractor,
    chunk_length_s      = 30,       # Whisper's native window
    stride_length_s     = 5,        # overlap for boundary stitching
    device              = 0 if DEVICE=="cuda" else -1,
    dtype               = torch.float16 if DEVICE=="cuda" else torch.float32,
    return_timestamps   = True,     # get word-level timestamps
)

def transcribe_wav(wav_path: Path, verbose: bool = True) -> dict:
    """
    Transcribe a WAV file using the ATC-fine-tuned Whisper model.
    
    Returns dict with:
        text        : full transcript string
        chunks      : list of {text, timestamp} dicts per segment
        duration_s  : audio duration in seconds
        rtf         : real-time factor (processing_time / audio_duration)
                      RTF < 1.0 means faster than real-time
    
    📚 Real-Time Factor (RTF):
        RTF = compute_time / audio_duration
        RTF = 0.1 means 10× faster than real-time (great for edge deployment)
        RTF = 1.0 means real-time (just barely acceptable)
        RTF > 1.0 means slower than real-time (unacceptable for live use)
        Target for our system: RTF < 0.3 on edge device
    """
    wav_path = Path(wav_path)
    
    # Get duration
    duration_s = librosa.get_duration(path=str(wav_path))
    
    if verbose:
        print(f"Transcribing: {wav_path.name}  ({duration_s:.1f}s)")
    
    t_start = time.time()
    
    # Load audio, resample to 16kHz (Whisper requirement)
    audio, _ = librosa.load(str(wav_path), sr=16000, mono=True)
    
    # Run ASR pipeline
    result = asr_pipeline(audio)
    
    elapsed = time.time() - t_start
    rtf = elapsed / duration_s
    
    # Clean up text
    text = result["text"].strip()
    # Remove common Whisper hallucinations on silence
    hallucinations = [
        "you", "thank you", "thanks for watching",
        "bye", "see you next time", "music"
    ]
    text_lower = text.lower()
    if any(h in text_lower for h in hallucinations) and len(text) < 30:
        text = ""
    
    if verbose:
        print(f"  ✅ Done in {elapsed:.1f}s  (RTF={rtf:.2f})")
        print(f"  Transcript: {text[:120]}{'...' if len(text)>120 else ''}")
    
    return {
        "wav_file"   : wav_path.name,
        "duration_s" : duration_s,
        "text"       : text,
        "chunks"     : result.get("chunks", []),
        "elapsed_s"  : elapsed,
        "rtf"        : rtf,
    }

print("✅ Transcription pipeline ready")
print(f"   chunk_length_s : 30s (Whisper native window)")
print(f"   stride_length_s: 5s  (boundary overlap)")
print(f"   return_timestamps: True (word-level timing)")

`torch_dtype` is deprecated! Use `dtype` instead!
Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


✅ Transcription pipeline ready
   chunk_length_s : 30s (Whisper native window)
   stride_length_s: 5s  (boundary overlap)
   return_timestamps: True (word-level timing)
